# 02 · Retrieval: top-k, scores, MMR and filters
**Concept:** the online half of the query pipeline: find the right chunks for a question.
**TAC scenario:** searching the indexed KB, release notes, tickets and guides.

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from rag_common import show, doc_ids, golden_set, chroma_filter

## 1. Similarity search with scores

In [ ]:
q = "IPsec tunnels to the warehouses drop for two minutes at the same time every night"
for doc, score in vectorstore.similarity_search_with_relevance_scores(q, k=5):
    print(f"{score:.3f}  [{doc.metadata['doc_id']:<10}] {doc.metadata['doc_type']:<12} {' '.join(doc.page_content.split())[:70]}")

## 2. Similarity vs MMR: relevance *and* diversity

In [ ]:
q = "router CPU spikes every few minutes when monitoring polls it"
sim = vectorstore.similarity_search(q, k=5)
mmr = vectorstore.max_marginal_relevance_search(q, k=5, fetch_k=25, lambda_mult=0.5)

print("SIMILARITY:"); show(sim)
print(f"→ {len(doc_ids(sim))} distinct documents\n")
print("MMR:"); show(mmr)
print(f"→ {len(doc_ids(mmr))} distinct documents")

## 3. Metadata filters: only the right product, version or type

In [ ]:
q = "tunnels drop overnight"
print("No filter:"); show(vectorstore.similarity_search(q, k=4))
f = chroma_filter(product="Nimbus NF-300", doc_type="kb")
print("\nFilter:", f); show(vectorstore.similarity_search(q, k=4, filter=f))

## 4. The retriever interface: `as_retriever()`

In [ ]:
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20})
docs = retriever.invoke("access points take ages to join the backup controller")
show(docs)

In [ ]:
# Retrievers are Runnables: batch works too
for q, docs in zip(["stack split after upgrade", "edge out of sync with orchestrator"],
                   retriever.batch(["stack split after upgrade", "edge out of sync with orchestrator"])):
    print(q, "→", doc_ids(docs))

## 5. Score threshold: return nothing when nothing is relevant

In [ ]:
thresh = vectorstore.as_retriever(search_type="similarity_score_threshold",
                                  search_kwargs={"score_threshold": 0.5, "k": 4})
for q in ["switch stack splits into two after upgrade", "What is the list price of a Helix HX-5000 48-port switch?"]:
    print(f"{q[:60]:60s} → {doc_ids(thresh.invoke(q)) or 'nothing above threshold'}")

> Scores depend on the embedding model; tune the threshold on your own data (Notebook 11 shows how to measure).

---
**🔧 TAC takeaway:** Search the KB, open the top few hits, skip duplicates, stay in your product line. Top-k, MMR and filters are those three habits in code.